# Scenario 7: Human Annotation and Review Workflow with Galileo

## What You'll Learn

By the end of this notebook you will know how to:

1. **Log traces** that are ready for human review
2. **Create annotation templates** — reusable review forms in five flavours: *score*, *star*, *tags*, *text*, and *thumbs up/down*
3. **Submit human ratings** against individual traces
4. **Inspect and export** annotations for downstream use

---

## Why Human Annotation?

Automated metrics (latency, token count, semantic similarity, etc.) are great for **scale**, but some qualities can only be judged by a human:

- **Empathy** — Does the response acknowledge the customer's frustration?
- **Brand voice** — Does the tone match your company's style guide?
- **Policy compliance** — Does the answer follow legal or regulatory rules?
- **Nuanced correctness** — Is the information technically accurate in a domain-specific way?

Annotations let human reviewers **score, tag, and comment** on traces directly inside Galileo, closing the loop between automated observability and human judgement.

---

## When to Use Annotations

| Use-Case | Example |
|---|---|
| QA teams reviewing production traces | Support-team leads spot-checking chatbot replies |
| Domain experts scoring outputs | A doctor verifying medical-advice accuracy |
| Building labeled datasets | Collecting human ratings to fine-tune a model or iterate on prompts |
| Compliance audits | Legal reviewers flagging responses that violate policy |

---

## Key Concepts

| Concept | What It Is |
|---|---|
| **Annotation Template** | A reusable review form — defines *what kind* of feedback to collect (e.g. a 1-5 score, a set of tags, free-form text). Created once, used across many traces. |
| **Rating** | A specific piece of feedback submitted by a reviewer against one trace (or span/session). |
| **Score** annotation | Numeric rating on a custom scale, e.g. 1–5. |
| **Star** annotation | 1–5 star rating — a familiar, quick quality signal. |
| **Tags** annotation | Categorical labels like `"too_vague"`, `"missing_empathy"`, `"unsafe"`. |
| **Text** annotation | Free-form reviewer note, e.g. *"Add empathy statement before refund."* |
| **Like / Dislike** annotation | Binary thumbs up / thumbs down — perfect for ship-it / do-not-ship decisions. |

> **Note:** This notebook uses the **Galileo REST API** directly for annotation operations (creating templates and submitting ratings). The SDK handles trace logging, but the annotation endpoints are HTTP calls — you'll see `requests.post` / `requests.put` in the code below.

---

## Prerequisites

- A Galileo account with a valid `GALILEO_API_KEY`
- An `OPENAI_API_KEY` (we reference a model name when logging spans)
- Python packages: `galileo`, `requests`, `python-dotenv`
- A `.env` file in the repo root containing both keys (see `.env.example`)

### Load Environment Variables

The cell below finds and loads your `.env` file, which must contain your `GALILEO_API_KEY` and `OPENAI_API_KEY`. It searches the current directory and the parent directory so the notebook works whether you run it from `agents/` or the repo root.

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


### Imports

A quick tour of what we're importing:

- **`requests`** — The standard Python HTTP library. We use it for the Galileo **annotation REST API** (creating templates, submitting ratings). The SDK doesn't wrap these endpoints yet, so we call them directly.
- **`GalileoLogger`**, **`Message`**, **`MessageRole`**, **`galileo_context`** — The core Galileo SDK objects for logging traces and spans.
- **`GalileoPythonConfig`** — Lets us read the current Galileo configuration (API URL, console URL, API key) so we can build REST requests.
- **`create_log_stream`**, **`get_log_stream`** — Helpers to create or retrieve a named log stream inside a project.
- **`delete_project`**, **`get_project`** — Project lifecycle helpers used during setup and cleanup.

In [ ]:
import requests

from galileo import GalileoLogger, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.log_streams import create_log_stream, get_log_stream
from galileo.projects import delete_project, get_project

PROJECT_NAME = 'GalileoEval_S7_Annotations'
LOG_STREAM_NAME = 'annotation-review'
MODEL = 'gpt-4o-mini'

## 1. Initialize Galileo

Every Galileo workflow starts by initializing a **project** and a **log stream**. The project is the top-level container (think of it as a folder), and the log stream is a named channel within that project where traces are stored.

After initialization the code prints clickable Console URLs so you can open the Galileo UI and follow along as traces and annotations are created.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

log_stream = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
if not log_stream:
    log_stream = create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')

## 2. Log Traces That Need Human Review

Before you can annotate anything, you need **traces to annotate**. This step logs three customer-support style traces — each one a prompt/response pair:

| # | Scenario | Why It's Interesting to Review |
|---|----------|-------------------------------|
| 1 | Refund email for a frustrated customer | Is it empathetic enough? |
| 2 | Delivery-delay update for a premium customer | Is it too vague? |
| 3 | Account-deletion confirmation | Is it clear and complete? |

The traces are intentionally **varied in quality** so that reviewers have interesting examples to rate.

**What the code does (pattern)**:

1. `logger.start_trace(input=...)` — opens a new trace
2. `logger.add_llm_span(...)` — records the LLM input/output inside the trace
3. Capture `trace_id` — this is the **unique identifier** that annotations will be attached to later
4. `logger.conclude(output=...)` — closes the trace
5. `logger.flush()` — sends all buffered traces to the Galileo backend

We store the trace IDs in `review_examples` so we can reference them when submitting ratings in Step 4.

In [4]:
review_samples = [
    {
        'input': 'Draft a refund email for a frustrated customer.',
        'output': 'We are sorry for the inconvenience. We will issue a refund within 5 business days.',
    },
    {
        'input': 'Write a short delivery-delay update for a premium customer.',
        'output': 'Your order is delayed. We will send another update soon.',
    },
    {
        'input': 'Respond to a user asking if their account deletion request is complete.',
        'output': 'Your account deletion request has been received and will be completed within 72 hours.',
    },
]

logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
review_examples = []

for sample in review_samples:
    logger.start_trace(input=sample['input'])
    logger.add_llm_span(
        input=[Message(role=MessageRole.user, content=sample['input'])],
        output=Message(role=MessageRole.assistant, content=sample['output']),
        model=MODEL,
    )
    trace_id = str(logger.traces[-1].id)
    logger.conclude(output=sample['output'])
    review_examples.append({**sample, 'trace_id': trace_id})

logger.flush()
review_examples


[{'input': 'Draft a refund email for a frustrated customer.',
  'output': 'We are sorry for the inconvenience. We will issue a refund within 5 business days.',
  'trace_id': 'da10c0ea-45f1-4483-9f53-1681911516f6'},
 {'input': 'Write a short delivery-delay update for a premium customer.',
  'output': 'Your order is delayed. We will send another update soon.',
  'trace_id': '0b2005d5-1b55-437a-8fb8-c595e4356ef1'},
 {'input': 'Respond to a user asking if their account deletion request is complete.',
  'output': 'Your account deletion request has been received and will be completed within 72 hours.',
  'trace_id': 'a6e71cee-3e33-4c6a-be4a-911d7bb376de'}]

## 3. Create Annotation Templates

Templates define the **review form** that human reviewers fill out. You create them once and reuse them across as many traces as you like.

This step creates **five** different template types to showcase the full range of annotation options:

| Template Name | Type | Purpose | Example Feedback |
|---|---|---|---|
| `helpfulness_review` | **score** (1–5) | Numeric quality rating | *"How helpful is this response? 4/5"* |
| `tone_star` | **star** (1–5) | Quick quality signal | *"Rate the tone: 4 stars"* |
| `issue_tags` | **tags** | Categorical issue labels | *Tag: `too_vague`, `missing_empathy`* |
| `review_note` | **text** | Free-form reviewer feedback | *"Add empathy statement before refund"* |
| `ship_it` | **like_dislike** | Binary ship / don't-ship decision | *Thumbs down — "needs revision"* |

**How it works under the hood:**

- Templates are created via the Galileo REST API: `POST /v2/projects/{project_id}/annotation/templates`
- The code is **idempotent** — it first fetches existing templates and only creates new ones if they don't already exist. So you can safely re-run this cell without duplicating templates.
- Each template includes a `criteria` field (a human-readable prompt that tells the reviewer what to evaluate) and a `constraints` field (defines the annotation type and any bounds like min/max for scores).

In [5]:
config = GalileoPythonConfig.get()
project = get_project(name=PROJECT_NAME)
project_id = str(project.id)
base_url = str(config.api_url).rstrip('/')
headers = {'Galileo-API-Key': config.api_key.get_secret_value()}

template_specs = [
    {
        'name': 'helpfulness_review',
        'criteria': 'Does the response help the customer clearly and politely?',
        'include_explanation': True,
        'constraints': {'annotation_type': 'score', 'min': 1, 'max': 5},
    },
    {
        'name': 'tone_star',
        'criteria': 'Rate the response tone from 1 to 5 stars.',
        'include_explanation': False,
        'constraints': {'annotation_type': 'star'},
    },
    {
        'name': 'issue_tags',
        'criteria': 'Tag quality issues you notice in the response.',
        'include_explanation': False,
        'constraints': {
            'annotation_type': 'tags',
            'tags': ['missing_empathy', 'too_vague', 'unsafe'],
            'allow_other': False,
        },
    },
    {
        'name': 'review_note',
        'criteria': 'Leave a reviewer note for follow-up.',
        'include_explanation': False,
        'constraints': {'annotation_type': 'text'},
    },
    {
        'name': 'ship_it',
        'criteria': 'Would you approve this response for production?',
        'include_explanation': True,
        'constraints': {'annotation_type': 'like_dislike'},
    },
]

existing_templates = requests.get(
    f'{base_url}/v2/projects/{project_id}/annotation/templates',
    headers=headers,
    timeout=30,
)
existing_templates.raise_for_status()
existing_by_name = {item['name']: item for item in existing_templates.json()}

annotation_templates = {}
for spec in template_specs:
    if spec['name'] in existing_by_name:
        annotation_templates[spec['name']] = existing_by_name[spec['name']]
        continue

    response = requests.post(
        f'{base_url}/v2/projects/{project_id}/annotation/templates',
        headers=headers,
        json=spec,
        timeout=30,
    )
    response.raise_for_status()
    annotation_templates[spec['name']] = response.json()

{
    name: {
        'id': template['id'],
        'annotation_type': template['constraints']['annotation_type'],
        'usage_count': template['usage_count'],
    }
    for name, template in annotation_templates.items()
}


{'helpfulness_review': {'id': '7077bb14-ac2a-48ae-a2e8-39b8b7e0dd33',
  'annotation_type': 'score',
  'usage_count': 3},
 'tone_star': {'id': 'a881a825-5f3d-48cb-81a1-334cfdace3cb',
  'annotation_type': 'star',
  'usage_count': 1},
 'issue_tags': {'id': 'd65d48f2-a4e5-4b14-a8bf-169d98c1f8c9',
  'annotation_type': 'tags',
  'usage_count': 1},
 'review_note': {'id': '1a063de2-eaf5-4e90-81a6-b36af1ed6101',
  'annotation_type': 'text',
  'usage_count': 1},
 'ship_it': {'id': 'f43f410e-7e66-4d32-9ff3-567ede5bbaf5',
  'annotation_type': 'like_dislike',
  'usage_count': 1}}

## 4. Submit Human Ratings to a Trace

A **rating** is one piece of feedback attached to one specific trace. Each rating references:

- A **template** (which defines the form — score, star, tags, etc.)
- A **record ID** (which trace to rate — here we use the `trace_id`)

In this step we annotate the **first trace** (the refund-email response) with all five template types, giving it a complete review:

| Template | Rating Value | Meaning |
|---|---|---|
| helpfulness_review | `4` | Helpful, but room for improvement |
| tone_star | `4 stars` | Good tone overall |
| issue_tags | `["missing_empathy"]` | Identified one issue |
| review_note | *"Add one sentence that validates..."* | Actionable suggestion |
| ship_it | `thumbs down` | Not ready for production yet |

> **In practice**, human reviewers would submit these ratings through the **Galileo Console UI** — a visual form that shows the trace and lets you click/type your feedback. Here we use the API directly to demonstrate the underlying data model.

**API endpoint:** `PUT /v2/projects/{project_id}/annotation/templates/{template_id}/records/{trace_id}/rating`

In [6]:
target_trace = review_examples[0]

rating_payloads = {
    'helpfulness_review': {
        'explanation': 'Polite and actionable, but it could acknowledge the customer frustration more explicitly.',
        'rating': {'annotation_type': 'score', 'value': 4},
    },
    'tone_star': {
        'rating': {'annotation_type': 'star', 'value': 4},
    },
    'issue_tags': {
        'rating': {'annotation_type': 'tags', 'value': ['missing_empathy']},
    },
    'review_note': {
        'rating': {'annotation_type': 'text', 'value': 'Add one sentence that validates the customer frustration before promising the refund.'},
    },
    'ship_it': {
        'explanation': 'Close to acceptable, but the tone needs one more revision before production.',
        'rating': {'annotation_type': 'like_dislike', 'value': False},
    },
}

annotation_results = {}
for template_name, body in rating_payloads.items():
    template_id = annotation_templates[template_name]['id']
    response = requests.put(
        f'{base_url}/v2/projects/{project_id}/annotation/templates/{template_id}/records/{target_trace["trace_id"]}/rating',
        headers=headers,
        json=body,
        timeout=30,
    )
    response.raise_for_status()
    annotation_results[template_name] = response.json()

{
    'trace_id': target_trace['trace_id'],
    'annotations': annotation_results,
}


{'trace_id': 'da10c0ea-45f1-4483-9f53-1681911516f6',
 'annotations': {'helpfulness_review': {'explanation': 'Polite and actionable, but it could acknowledge the customer frustration more explicitly.',
   'rating': {'annotation_type': 'score', 'value': 4},
   'created_at': '2026-03-14T03:21:37.000716Z',
   'created_by': '7680b8ce-69f8-40ea-8238-d7abcc80149f'},
  'tone_star': {'explanation': None,
   'rating': {'annotation_type': 'star', 'value': 4},
   'created_at': '2026-03-14T03:21:38.506334Z',
   'created_by': '7680b8ce-69f8-40ea-8238-d7abcc80149f'},
  'issue_tags': {'explanation': None,
   'rating': {'annotation_type': 'tags', 'value': ['missing_empathy']},
   'created_at': '2026-03-14T03:21:39.839477Z',
   'created_by': '7680b8ce-69f8-40ea-8238-d7abcc80149f'},
  'review_note': {'explanation': None,
   'rating': {'annotation_type': 'text',
    'value': 'Add one sentence that validates the customer frustration before promising the refund.'},
   'created_at': '2026-03-14T03:21:41.2471

## 5. Inspect the Templates and Next Steps

After submitting ratings you can verify that the `usage_count` on each template has increased — this confirms your annotations were recorded.

**What you can do in the Galileo Console:**

- **Logs view** — Annotations appear as extra columns alongside your traces. You can sort and filter by annotation values (e.g. show only traces with a helpfulness score below 3).
- **Messages view** — Shows the full conversation and lets reviewers submit annotations through a visual form (no code required).
- **Annotation Queues** *(Enterprise Beta)* — Route specific traces to specific reviewers automatically, useful for large-scale review workflows.
- **Export** — Download annotated data as CSV/JSON. Use exported annotations to build training datasets, improve prompts, or track quality trends over time.

In [7]:
template_list = requests.get(
    f'{base_url}/v2/projects/{project_id}/annotation/templates',
    headers=headers,
    timeout=30,
)
template_list.raise_for_status()

[
    {
        'name': template['name'],
        'annotation_type': template['constraints']['annotation_type'],
        'usage_count': template['usage_count'],
    }
    for template in template_list.json()
]


[{'name': 'helpfulness_review', 'annotation_type': 'score', 'usage_count': 4},
 {'name': 'issue_tags', 'annotation_type': 'tags', 'usage_count': 2},
 {'name': 'review_note', 'annotation_type': 'text', 'usage_count': 2},
 {'name': 'ship_it', 'annotation_type': 'like_dislike', 'usage_count': 2},
 {'name': 'tone_star', 'annotation_type': 'star', 'usage_count': 2}]

## 6. Optional Cleanup

You may want to **keep the project** so you can explore the annotations in the Galileo Console — click the URLs printed in Step 1 to see your traces and ratings in the UI.

When you're finished exploring, run the cell below to delete the demo project and all its data.

In [ ]:
try:
    delete_project(name=PROJECT_NAME)
except Exception:
    pass

'Deleted annotation demo project'
